# Apprentissage des bounding boxes

Ce notebook reprend le modèle `satellite_debris_plateau_70_retrain.keras`, déjà entraîné pour distinguer `satellite` et `debris`, puis ajoute une sortie `bbox_output` pour apprendre la boîte englobante.

Les labels YOLO sont convertis en `[ymin, xmin, ymax, xmax]`, avec des coordonnées normalisées entre `0` et `1`. Chaque image doit contenir un seul objet.

In [1]:
import glob
import logging
import os

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import pandas as pd
import tensorflow as tf
import yaml

from logging_settings import setup_logging
from tensorflow import keras
from tensorflow.keras import layers

setup_logging(level=logging.INFO, log_file="bbox_training", console=True, enable_traceback=True)

IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 60
MODEL_PATH = "models/satellite_debris_plateau_70_retrain.keras"
OUTPUT_PATH = "models_bbox/satellite_debris_bbox_hardcoded.keras"

# Choix possibles : hardcoded, plateau ou cosine_decay.
# Exemple : LEARNING_RATE_MODE=plateau jupyter notebook bounding_box_training.ipynb

LEARNING_RATE_MODE = "cosine_decay"  
INITIAL_LEARNING_RATE = 1e-3



LEARNING_RATE_MODE = {
    "hardcode": "hardcoded",
    "cosine": "cosine_decay",
}.get(LEARNING_RATE_MODE, LEARNING_RATE_MODE)

if LEARNING_RATE_MODE not in {"cosine_decay", "plateau", "hardcoded"}:
    raise ValueError(
        "LEARNING_RATE_MODE doit être 'hardcoded', 'plateau' ou 'cosine_decay'"
    )

GPU_DEVICES = tf.config.list_physical_devices("GPU")
if GPU_DEVICES:
    for gpu in GPU_DEVICES:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError:
            pass
    TRAINING_DEVICE = "/GPU:0"
    logging.info("Entraînement sur GPU : %s", [gpu.name for gpu in GPU_DEVICES])
else:
    TRAINING_DEVICE = "/CPU:0"
    logging.warning("Aucun GPU détecté : entraînement sur CPU")

with open("data/data.yaml", "r", encoding="utf-8") as yaml_file:
    dataset_config = yaml.safe_load(yaml_file)

ORIGINAL_CLASS_NAMES = dataset_config["names"]
CLASS_NAMES = ["satellite", "debris"]
logging.info("TensorFlow : %s", tf.__version__)
logging.info("Classes originales : %s", ORIGINAL_CLASS_NAMES)
logging.info("Classes du modèle : %s", CLASS_NAMES)

I0000 00:00:1792503200.733274  436787 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792503200.758586  436787 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI AVX_VNNI_INT8 AVX_NE_CONVERT FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792503201.277618  436787 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-10-20 15:33:21 [INFO] : root : Entraînement sur GPU : ['/physical_device:GPU:0']
W0000 00:00:1792503

## Chargement des labels YOLO

Chaque ligne de label est `class_id x_center y_center width height`. On utilise la première ligne, car le dataset actuel contient un objet par image.

In [2]:
def create_dataset_dataframe(image_dir):
    records = []
    image_paths = sorted(glob.glob(os.path.join(image_dir, "*.jpg")))

    for image_path in image_paths:
        label_path = image_path.replace("images", "labels").replace(".jpg", ".txt")
        if not os.path.exists(label_path):
            continue

        with open(label_path, "r", encoding="utf-8") as label_file:
            line = label_file.readline().strip()
        if not line:
            continue

        parts = line.split()
        if len(parts) != 5:
            raise ValueError(f"Label YOLO invalide : {label_path}")

        class_id = int(parts[0])
        x_center, y_center, width, height = map(float, parts[1:])
        class_name = ORIGINAL_CLASS_NAMES[class_id]
        ymin = max(0.0, y_center - height / 2.0)
        xmin = max(0.0, x_center - width / 2.0)
        ymax = min(1.0, y_center + height / 2.0)
        xmax = min(1.0, x_center + width / 2.0)

        binary_class_id = int(class_name == "debris")
        records.append({
            "image_path": image_path,
            "binary_class_id": binary_class_id,
            "binary_class_name": CLASS_NAMES[binary_class_id],
            "ymin": ymin,
            "xmin": xmin,
            "ymax": ymax,
            "xmax": xmax,
        })

    return pd.DataFrame(records)

df_train = create_dataset_dataframe("data/train/images")
df_val = create_dataset_dataframe("data/valid/images")
df_test = create_dataset_dataframe("data/test/images")
logging.info("Train : %d images", len(df_train))
logging.info("Val   : %d images", len(df_val))
logging.info("Test  : %d images", len(df_test))
logging.info("Répartition train :\n%s", df_train["binary_class_name"].value_counts())
df_train.head()

2026-10-20 15:33:21 [INFO] : root : Train : 2105 images
2026-10-20 15:33:21 [INFO] : root : Val   : 239 images
2026-10-20 15:33:21 [INFO] : root : Test  : 123 images
2026-10-20 15:33:21 [INFO] : root : Répartition train :
binary_class_name
satellite    1919
debris        186
Name: count, dtype: int64


,image_path,binary_class_id,binary_class_name,ymin,xmin,ymax,xmax
0,data/train/images/img000008_jpg.rf.2c95cb6fb03...,0,satellite,0.369141,0.129297,0.865234,0.430078
1,data/train/images/img000008_jpg.rf.9f76dc15cc1...,0,satellite,0.134766,0.129297,0.630859,0.430078
2,data/train/images/img000009_jpg.rf.1142d9cc183...,0,satellite,0.341016,0.192578,0.969922,0.585547
3,data/train/images/img000009_jpg.rf.1c26d8ebd8e...,0,satellite,0.341016,0.414453,0.969922,0.807422
4,data/train/images/img000009_jpg.rf.da7b45fe680...,0,satellite,0.030078,0.192578,0.658984,0.585547


In [3]:
def df_to_multitask_dataset(df, is_training):
    image_paths = df["image_path"].values
    boxes = df[["ymin", "xmin", "ymax", "xmax"]].values.astype(np.float32)
    class_ids = df["binary_class_id"].values.astype(np.float32)
    dataset = tf.data.Dataset.from_tensor_slices((image_paths, boxes, class_ids))

    def load_and_preprocess(image_path, box, class_id):
        image = tf.io.read_file(image_path)
        image = tf.image.decode_jpeg(image, channels=3)
        image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
        image = keras.applications.mobilenet_v2.preprocess_input(image)
        return image, {
            "class_output": tf.expand_dims(class_id, axis=-1),
            "bbox_output": box,
        }

    if is_training:
        dataset = dataset.shuffle(min(len(df), 1000), reshuffle_each_iteration=True)
    return dataset.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = df_to_multitask_dataset(df_train, is_training=True)
val_ds = df_to_multitask_dataset(df_val, is_training=False)
test_ds = df_to_multitask_dataset(df_test, is_training=False)

W0000 00:00:1792503201.873750  436787 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
I0000 00:00:1792503201.954954  436787 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 29228 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5090, pci bus id: 0000:02:00.0, compute capability: 12.0a


## Création du modèle multitâche

On extrait le backbone MobileNetV2 déjà appris par l’ancien modèle. La nouvelle tête apprend simultanément la classe et les quatre coordonnées de la boîte.

## Comment juger si une boîte prédite est bonne ?

La loss (Huber) et la MAE mesurent l'écart moyen des 4 coordonnées, mais ne disent pas directement si la boîte « tombe sur l'objet ». Le critère standard en détection d'objets est l'**IoU** (*Intersection over Union*) :

$$\text{IoU} = \frac{\text{aire}(\text{vraie boîte} \cap \text{boîte prédite})}{\text{aire}(\text{vraie boîte} \cup \text{boîte prédite})}$$

- `IoU = 1` : les deux boîtes sont identiques ;
- `IoU = 0` : elles ne se touchent pas ;
- par convention (PASCAL VOC), une boîte est **bonne si IoU ≥ 0.5**. Des benchmarks plus stricts (COCO) moyennent aussi les seuils de 0.5 à 0.95.

Une MAE de 0.12 peut paraître faible, mais sur un petit objet un décalage de 12 % de l'image suffit à donner une IoU proche de 0. C'est pourquoi l'IoU est ajoutée comme métrique (`bbox_output_iou`) et sert à choisir le meilleur modèle.

Une **détection complète** est correcte quand la classe prédite est la bonne **et** que l'IoU ≥ 0.5.


In [4]:
old_model = keras.models.load_model(MODEL_PATH, compile=False)
backbone = old_model.get_layer("mobilenetv2_1.00_224")
backbone.trainable = True

inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="image_input")
features = backbone(inputs, training=False)
features = layers.GlobalAveragePooling2D()(features)
features = layers.Dropout(0.2)(features)

class_output = layers.Dense(1, activation="sigmoid", name="class_output")(features)
bbox_output = layers.Dense(4, activation="sigmoid", name="bbox_output")(features)

model = keras.Model(
    inputs=inputs,
    outputs={"class_output": class_output, "bbox_output": bbox_output},
)

# Une boîte prédite est jugée correcte si son IoU avec la vraie boîte atteint ce seuil
IOU_THRESHOLD = 0.5

def iou(y_true, y_pred):
    # Boîtes au format [ymin, xmin, ymax, xmax], normalisées entre 0 et 1
    inter_ymin = keras.ops.maximum(y_true[..., 0], y_pred[..., 0])
    inter_xmin = keras.ops.maximum(y_true[..., 1], y_pred[..., 1])
    inter_ymax = keras.ops.minimum(y_true[..., 2], y_pred[..., 2])
    inter_xmax = keras.ops.minimum(y_true[..., 3], y_pred[..., 3])
    intersection = (
        keras.ops.maximum(0.0, inter_ymax - inter_ymin)
        * keras.ops.maximum(0.0, inter_xmax - inter_xmin)
    )
    true_area = (y_true[..., 2] - y_true[..., 0]) * (y_true[..., 3] - y_true[..., 1])
    # Les 4 sorties sont indépendantes : une boîte prédite peut être "retournée"
    # (ymax < ymin), on lui donne alors une aire nulle.
    pred_area = (
        keras.ops.maximum(0.0, y_pred[..., 2] - y_pred[..., 0])
        * keras.ops.maximum(0.0, y_pred[..., 3] - y_pred[..., 1])
    )
    union = true_area + pred_area - intersection
    return intersection / (union + 1e-7)

def box_accuracy(y_true, y_pred):
    # 1 si la boîte est bonne (IoU >= seuil), 0 sinon : la moyenne Keras
    # donne le ratio boîtes bonnes / nombre total de boîtes
    return keras.ops.cast(iou(y_true, y_pred) >= IOU_THRESHOLD, "float32")

def compile_multitask_model(model, learning_rate=INITIAL_LEARNING_RATE):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss={
            "class_output": keras.losses.BinaryCrossentropy(),
            "bbox_output": keras.losses.Huber(),
        },
        loss_weights={"class_output": 1.0, "bbox_output": 5.0},
        metrics={
            "class_output": [keras.metrics.BinaryAccuracy(name="accuracy")],
            "bbox_output": [keras.metrics.MeanAbsoluteError(name="mae"), iou, box_accuracy],
        },
    )

if LEARNING_RATE_MODE == "cosine_decay":
    # Nombre exact de batches par epoch (le dernier batch incomplet est conservé)
    total_steps = len(train_ds) * EPOCHS
    learning_rate = keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=INITIAL_LEARNING_RATE,
        decay_steps=total_steps,
        alpha=1e-2,
    )
else:
    # Utilisé directement pour le mode hardcoded et comme valeur initiale
    # lorsque ReduceLROnPlateau ajuste le taux pendant l'entraînement.
    learning_rate = INITIAL_LEARNING_RATE

compile_multitask_model(model, learning_rate)
logging.info(
    f"Mode du learning rate : {LEARNING_RATE_MODE} "
    f"(initial={INITIAL_LEARNING_RATE:f}, epochs={EPOCHS})"
)
model.summary()

2026-10-20 15:33:22 [INFO] : root : Mode du learning rate : cosine_decay (initial=0.001000, epochs=60)


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image_input         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mobilenetv2_1.00_2… │ (None, 7, 7,      │  2,257,984 │ image_input[0][0] │
│ (Functional)        │ 1280)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 1280)      │          0 │ mobilenetv2_1.00… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 1280)      │          0 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bbox_output (Dense) │ (None, 4)         │      5,124 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ class_output        │ (None, 1)         │      1,281 │ dropout[0][0]     │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,264,389 (8.64 MB)

 Trainable params: 2,230,277 (8.51 MB)

 Non-trainable params: 34,112 (133.25 KB)

## Entraînement

Les augmentations géométriques sont volontairement absentes ici : une rotation ou un retournement doit aussi transformer les coordonnées de la boîte.

In [ ]:
class EpochBBoxPreviewCallback(keras.callbacks.Callback):
    def __init__(self, dataset, class_names, threshold=0.5):
        super().__init__()
        self.class_names = class_names
        # Probabilité de debris à partir de laquelle l'objet est classé debris
        self.threshold = threshold
        self.images = []
        self.true_boxes = []
        self.true_class_ids = []

        for images, targets in dataset:
            self.images.extend(images.numpy())
            self.true_boxes.extend(targets["bbox_output"].numpy())
            self.true_class_ids.extend(targets["class_output"].numpy()[:, 0].astype(int))

    def on_epoch_end(self, epoch, logs=None):
        index = np.random.randint(len(self.images))
        image = self.images[index][np.newaxis, ...]
        true_class_name = self.class_names[self.true_class_ids[index]]

        predictions = self.model.predict(image, verbose=0)
        class_probability = float(predictions["class_output"][0, 0])
        box_iou = float(iou(self.true_boxes[index], predictions["bbox_output"][0]))
        box_verdict = "bonne" if box_iou >= IOU_THRESHOLD else "mauvaise"
        predicted_class_name = self.class_names[int(class_probability >= self.threshold)]

        logging.info(
            f"Epoch {epoch + 1} - Réel : {true_class_name} "
            f"| Prévu : {predicted_class_name} (p_debris={class_probability:.2f}) "
            f"| IoU={box_iou:.2f} -> boîte {box_verdict}"
        )

        figure, axis = plt.subplots(figsize=(5, 5))
        # MobileNetV2 attend des pixels dans [-1, 1]. Pour l'affichage,
        # reconvertit uniquement une copie vers l'intervalle RGB [0, 1].
        display_image = np.clip((image[0] + 1.0) / 2.0, 0.0, 1.0)
        height, width = display_image.shape[:2]
        axis.imshow(display_image)

        boxes = [
            (self.true_boxes[index], "lime", "Réel"),
            (predictions["bbox_output"][0], "red", "Prévu"),
        ]
        for (ymin, xmin, ymax, xmax), color, label in boxes:
            axis.add_patch(patches.Rectangle(
                (xmin * width, ymin * height),
                (xmax - xmin) * width,
                (ymax - ymin) * height,
                linewidth=2,
                edgecolor=color,
                facecolor="none",
                label=label,
            ))

        axis.set_title(
            f"Epoch {epoch + 1} | Réel : {true_class_name} "
            f"| Prévu : {predicted_class_name} ({class_probability:.2f})\n"
            f"IoU = {box_iou:.2f} -> boîte {box_verdict}"
        )
        axis.legend(loc="upper right")
        axis.axis("off")
        figure.tight_layout()
        plt.show()
        plt.close(figure)


early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_bbox_output_iou",
    mode="max",
    # Plus long que la patience de ReduceLROnPlateau pour laisser
    # le temps à un learning rate réduit d'améliorer le modèle.
    patience=220,
    restore_best_weights=True,
)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        OUTPUT_PATH,
        monitor="val_bbox_output_iou",
        mode="max",
        save_best_only=True,
    ),
    early_stopping,
    EpochBBoxPreviewCallback(val_ds, CLASS_NAMES),
]

if LEARNING_RATE_MODE == "plateau":
    callbacks.append(
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_bbox_output_iou",
            mode="max",
            factor=0.5,
            patience=5,
            min_lr=1e-6,
            verbose=1,
        )
    )

with tf.device(TRAINING_DEVICE):
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=callbacks,
    )

if early_stopping.stopped_epoch > 0:
    logging.warning(
        f"Entraînement arrêté par EarlyStopping à l'epoch {early_stopping.stopped_epoch + 1}/{EPOCHS} : "
        f"{early_stopping.monitor} ne s'est pas amélioré depuis {early_stopping.patience} epochs. "
        f"Meilleure valeur : {early_stopping.best:.4f} à l'epoch {early_stopping.best_epoch + 1}, "
        f"poids de cette epoch restaurés (et sauvegardés dans {OUTPUT_PATH})."
    )


Epoch 1/60


/home/jzory/Documents/Leo/.venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
I0000 00:00:1792503209.287189  436956 service.cc:153] XLA service 0x7b02740b89f0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1792503209.287205  436956 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 5090, Compute Capability 12.0a (Driver: 13.2.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.27.0)
I0000 00:00:1792503209.510015  436956 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1792503210.787805  436956 cuda_dnn.cc:461] Loaded cuDNN version 92700
I0000 00:00:1792503211.000749  436956 dot_merger.cc:481] Mer

 11/132 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - bbox_output_box_accuracy: 0.1136 - bbox_output_iou: 0.2218 - bbox_output_loss: 0.0246 - bbox_output_mae: 0.1757 - class_output_accuracy: 0.9375 - class_output_loss: 0.2057 - loss: 0.3285

I0000 00:00:1792503223.570650  436956 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


131/132 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - bbox_output_box_accuracy: 0.2238 - bbox_output_iou: 0.3242 - bbox_output_loss: 0.0139 - bbox_output_mae: 0.1307 - class_output_accuracy: 0.9876 - class_output_loss: 0.0417 - loss: 0.1112

I0000 00:00:1792503226.543316  436957 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_33945__.329
E0000 00:00:1792503227.667959  436957 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1792503228.634404  436957 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1792503229.365108  436957 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step - bbox_output_box_accuracy: 0.2242 - bbox_output_iou: 0.3245 - bbox_output_loss: 0.0139 - bbox_output_mae: 0.1306 - class_output_accuracy: 0.9876 - class_output_loss: 0.0414 - loss: 0.1109

E0000 00:00:1792503238.985812  436955 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1792503239.454629  436955 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1792503239.840423  436955 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


In [ ]:
best_model = keras.models.load_model(OUTPUT_PATH, compile=False)
compile_multitask_model(best_model)
test_metrics = best_model.evaluate(test_ds, return_dict=True)
logging.info("Métriques test : %s", test_metrics)

## Visualisation d'une prédiction

Le modèle renvoie une probabilité de débris et une boîte normalisée `[ymin, xmin, ymax, xmax]`.

In [ ]:
images, targets = next(iter(test_ds))
predictions = best_model.predict(images, verbose=0)
index = 0
image = images[index].numpy()
image_for_display = (image + 1.0) / 2.0
class_probability = float(predictions["class_output"][index, 0])
ymin, xmin, ymax, xmax = predictions["bbox_output"][index]
height, width = image.shape[:2]

figure, axis = plt.subplots(figsize=(7, 7))
axis.imshow(np.clip(image_for_display, 0.0, 1.0))
axis.add_patch(patches.Rectangle(
    (xmin * width, ymin * height),
    (xmax - xmin) * width,
    (ymax - ymin) * height,
    linewidth=2,
    edgecolor="lime",
    facecolor="none",
))
axis.set_title(f"Prédit : {CLASS_NAMES[int(class_probability >= 0.5)]} ({class_probability:.2f})")
axis.axis("off")
plt.show()

## Courbes d'apprentissage

Évolution de la loss, de l'IoU (avec le seuil 0.5 en pointillés), de la MAE des coordonnées, de l'accuracy de classification et de l'accuracy des boîtes (boîtes bonnes, IoU ≥ 0.5, / nombre total).


In [ ]:
metrics = history.history

figure, axes = plt.subplots(2, 3, figsize=(20, 10))

for axis, (metric_name, title) in zip(axes.flatten(), [
    ("loss", "Loss totale (classe + 5 x bbox)"),
    ("bbox_output_iou", "IoU moyenne des boîtes"),
    ("bbox_output_mae", "MAE des coordonnées de boîte"),
    ("class_output_accuracy", "Accuracy de classification"),
    ("bbox_output_box_accuracy", f"Accuracy des boîtes (IoU >= {IOU_THRESHOLD})"),
]):
    axis.plot(history.epoch, metrics[metric_name], "o-", label="train")
    axis.plot(history.epoch, metrics[f"val_{metric_name}"], "o-", label="validation")
    if metric_name == "bbox_output_iou":
        axis.axhline(IOU_THRESHOLD, color="gray", linestyle="--", label=f"seuil {IOU_THRESHOLD}")
    axis.set_title(title)
    axis.set_xlabel("Epoch")
    axis.legend()

# 5 courbes pour 6 emplacements : on masque le dernier
axes.flatten()[-1].axis("off")

plt.tight_layout()
plt.show()


## Évaluation sur le jeu de test

Matrice de confusion pour la classe, distribution des IoU pour les boîtes, et taux de détections complètes correctes (bonne classe et IoU ≥ 0.5).


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix

y_test = []
true_boxes_test = []
p_test = []
pred_boxes_test = []
for images, targets in test_ds:
    predictions = best_model.predict(images, verbose=0)
    y_test.append(targets["class_output"].numpy()[:, 0].astype(int))
    true_boxes_test.append(targets["bbox_output"].numpy())
    p_test.append(predictions["class_output"][:, 0])
    pred_boxes_test.append(predictions["bbox_output"])

y_test = np.concatenate(y_test)
true_boxes_test = np.concatenate(true_boxes_test)
p_test = np.concatenate(p_test)
pred_boxes_test = np.concatenate(pred_boxes_test)

pred_test = (p_test >= 0.5).astype(int)
iou_test = keras.ops.convert_to_numpy(iou(true_boxes_test, pred_boxes_test))
good_box = iou_test >= IOU_THRESHOLD
good_detection = good_box & (pred_test == y_test)

logging.info("Classification sur le test :\n%s", classification_report(y_test, pred_test, target_names=CLASS_NAMES, digits=4))
logging.info("IoU test : moyenne=%.3f, médiane=%.3f", iou_test.mean(), np.median(iou_test))
logging.info("Boîtes correctes (IoU >= %.1f) : %.1f %%", IOU_THRESHOLD, 100 * good_box.mean())
logging.info("Détections correctes (classe + boîte) : %.1f %%", 100 * good_detection.mean())
for class_id, class_name in enumerate(CLASS_NAMES):
    mask = y_test == class_id
    if mask.any():
        logging.info("  %s : IoU moyenne=%.3f, boîtes correctes=%.1f %%",
                     class_name, iou_test[mask].mean(), 100 * good_box[mask].mean())

figure, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_test, pred_test, labels=[0, 1])
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES).plot(cmap="Blues", ax=axes[0], colorbar=False)
axes[0].set_title("Matrice de confusion - test")

for class_id, class_name in enumerate(CLASS_NAMES):
    axes[1].hist(iou_test[y_test == class_id], bins=20, range=(0, 1), alpha=0.6, label=class_name)
axes[1].axvline(IOU_THRESHOLD, color="gray", linestyle="--", label=f"seuil {IOU_THRESHOLD}")
axes[1].set_title(f"Distribution des IoU - test ({100 * good_box.mean():.1f} % >= {IOU_THRESHOLD})")
axes[1].set_xlabel("IoU")
axes[1].set_ylabel("Nombre d'images")
axes[1].legend()

plt.tight_layout()
plt.show()


Le nouveau modèle est enregistré dans `models_bbox/satellite_debris_bbox.keras`. Pour le recharger plus tard :

```python
model = keras.models.load_model("models_bbox/satellite_debris_bbox.keras", compile=False)
```